# 01 - Pull Statcast data

**Goal:** download pitch-level Statcast data for each regular season in `SEASONS` (see `src/config.py`) and run basic checks.

Raw files are saved to `data/raw/` (git-ignored). Only small summary tables go to `data/processed/` so they can be committed.

In [2]:
import sys
sys.path.append("..")  # lets the notebook import from src/

import pandas as pd
from src.config import SEASONS, PROCESSED
from src.data import pull_season, load_seasons, add_park

SEASONS

[2025]

## Pull each season

The first run is slow (a full season can take 10-20+ minutes). Each season is saved as a parquet file, and seasons that already exist are skipped, so it is safe to re-run.

In [9]:
for year in [2022, 2023, 2024, 2026]:
    pull_season(year)

2022: already downloaded, skipping
2023: already downloaded, skipping
2024: already downloaded, skipping
2026: already downloaded, skipping


## Load and label parks

In [15]:
df = add_park(load_seasons())
print(df.shape)
df.head()

(712528, 42)


,game_pk,game_date,game_year,game_type,home_team,away_team,inning_topbot,at_bat_number,pitch_number,pitcher,...,strikes,launch_speed,launch_angle,hit_distance_sc,hc_x,hc_y,woba_value,woba_denom,estimated_woba_using_speedangle,park
0,776151,2025-09-28,2025,R,BOS,DET,Top,74,1,669711,...,0,83.8,37,290,96.06,86.56,0.0,1,0.009,Fenway Park
1,776151,2025-09-28,2025,R,BOS,DET,Top,73,4,669711,...,2,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,1,0.0,Fenway Park
2,776151,2025-09-28,2025,R,BOS,DET,Top,73,3,669711,...,2,80.3,45,246,<NA>,<NA>,<NA>,<NA>,<NA>,Fenway Park
3,776151,2025-09-28,2025,R,BOS,DET,Top,73,2,669711,...,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Fenway Park
4,776151,2025-09-28,2025,R,BOS,DET,Top,73,1,669711,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Fenway Park


## Check what is populated

`arm_angle` is the one to watch. If it is mostly empty in early seasons, either start the analysis window later (edit `SEASONS` in `src/config.py`) or fall back to release height (the archetype code can do this and tells you which source it used).

In [13]:
key = ["release_speed", "release_spin_rate", "pfx_x", "pfx_z", "release_pos_x", "release_pos_z",
       "release_extension", "arm_angle", "launch_speed", "estimated_woba_using_speedangle"]
key = [c for c in key if c in df.columns]
df.groupby("game_year")[key].agg(lambda s: s.isna().mean()).round(3)

,release_speed,release_spin_rate,pfx_x,pfx_z,release_pos_x,release_pos_z,release_extension,arm_angle,launch_speed,estimated_woba_using_speedangle
game_year,,,,,,,,,,
2025,0.004,0.006,0.004,0.004,0.004,0.004,0.004,0.005,0.669,0.746


## Check parks and games

Each park should have about 81 home games per season (a few fewer if games were never made up). Spot-check anything unusual, such as a team that changed stadiums. Neutral-site games (international series) get labeled with the designated home team's park by this simple mapping. It is only a handful of games per year, but if you want to remove them, look up their `game_pk` values and add them to `NEUTRAL_SITE_GAME_PKS` in `src/data.py`.

In [7]:
coverage = (df.groupby(["game_year", "park"])
              .agg(pitches=("pitch_type", "size"), games=("game_pk", "nunique"))
              .reset_index())
coverage.query("games != 81").sort_values("games")

,game_year,park,pitches,games


In [8]:
coverage.to_csv(PROCESSED / "data_coverage.csv", index=False)

## Commit

```
git add notebooks/01_pull_data.ipynb data/processed/data_coverage.csv
git commit -m "add data pull and coverage checks"
git push
```